# AI Skill Gap & Career Recommendation System

## Model Used
Random Forest Classifier

## Dataset
`tgtdataset.csv`

## Goal
Predict suitable job roles based on:
- age
- education level
- current skills

Also recommend:
- skill gaps
- training programs

## Import Libraries

In [65]:
import pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MultiLabelBinarizer

from sklearn.model_selection import train_test_split

import xgboost as xgb
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

from joblib import dump

## Load Dataset

In [66]:
df = pd.read_csv('tgtdataset.csv')

df.columns = df.columns.str.strip()

print('Dataset Shape:', df.shape)

print('\nDataset Preview:\n')
print(df.head())

Dataset Shape: (1500, 11)

Dataset Preview:

   age  gender    location education_level internet_access  income_decile  \
0   19  Female       Urban       Secondary             Yes              2   
1   52  Female       Urban      Vocational             Yes              1   
2   24    Male       Rural         Primary             Yes              6   
3   54    Male  Semi-Urban         Primary             Yes              4   
4   28  Female       Rural       Secondary             Yes              2   

  current_sector                          current_skills  \
0   Construction              Communication, Tool Repair   
1       Services                 Typing, Email Etiquette   
2         Retail     Sales, Communication, Typing, Excel   
3         Retail  Sales, Communication, Inventory Basics   
4         Retail         Communication, Inventory Basics   

              target_job_role                    predicted_skill_gap  \
0           Certified Plumber         Pipe Fitting, Safety 

## Clean Skills Column

In [67]:
def clean_skills(x):

    return [skill.strip() for skill in str(x).split(',')]

df['current_skills'] = df['current_skills'].apply(clean_skills)

print(df['current_skills'].head())

0                [Communication, Tool Repair]
1                   [Typing, Email Etiquette]
2       [Sales, Communication, Typing, Excel]
3    [Sales, Communication, Inventory Basics]
4           [Communication, Inventory Basics]
Name: current_skills, dtype: object


## Encode Skills

In [68]:
mlb = MultiLabelBinarizer()

skills_encoded = mlb.fit_transform(df['current_skills'])

skills_df = pd.DataFrame(
    skills_encoded,
    columns=mlb.classes_
)

print('Total Skills:', len(mlb.classes_))

print(skills_df.head())

Total Skills: 33
   Accounting  Basic Coding  Basic Math  Basic Python  Bookkeeping  CSS  \
0           0             0           0             0            0    0   
1           0             0           0             0            0    0   
2           0             0           0             0            0    0   
3           0             0           0             0            0    0   
4           0             0           0             0            0    0   

   Communication  Construction Safety  Customer Handling  Data Entry  ...  \
0              1                    0                  0           0  ...   
1              0                    0                  0           0  ...   
2              1                    0                  0           0  ...   
3              1                    0                  0           0  ...   
4              1                    0                  0           0  ...   

   Pipe Fitting  Plumbing  React  Retail Operations  Sales  Soil Test

## Select Features

In [69]:
main_df = df[['age', 'education_level']].copy()

print(main_df.head())

   age education_level
0   19       Secondary
1   52      Vocational
2   24         Primary
3   54         Primary
4   28       Secondary


## Encode Education Level

In [70]:
education_encoder = LabelEncoder()

main_df['education_level'] = education_encoder.fit_transform(
    main_df['education_level']
)

print(main_df.head())

   age  education_level
0   19                2
1   52                3
2   24                1
3   54                1
4   28                2


## Encode Target Job Roles

In [71]:
target_encoder = LabelEncoder()

y = target_encoder.fit_transform(df['target_job_role'])

print(target_encoder.classes_)

['Certified Plumber' 'Customer Support Executive' 'Data Entry Operator'
 'Junior Accountant' 'Modern Farm Manager' 'Retail Store Manager'
 'Sales Associate' 'Web Developer']


## Create Final Feature Matrix

In [72]:
X = pd.concat(
    [
        main_df.reset_index(drop=True),
        skills_df.reset_index(drop=True)
    ],
    axis=1
)

print('Feature Matrix Shape:', X.shape)

print(X.head())

Feature Matrix Shape: (1500, 35)
   age  education_level  Accounting  Basic Coding  Basic Math  Basic Python  \
0   19                2           0             0           0             0   
1   52                3           0             0           0             0   
2   24                1           0             0           0             0   
3   54                1           0             0           0             0   
4   28                2           0             0           0             0   

   Bookkeeping  CSS  Communication  Construction Safety  ...  Pipe Fitting  \
0            0    0              1                    0  ...             0   
1            0    0              0                    0  ...             0   
2            0    0              1                    0  ...             0   
3            0    0              1                    0  ...             0   
4            0    0              1                    0  ...             0   

   Plumbing  React  Ret

## Save Training Columns

In [73]:
training_columns = X.columns.tolist()

## Train Test Split

In [74]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Train Shape:', X_train.shape)
print('Test Shape:', X_test.shape)

Train Shape: (1200, 35)
Test Shape: (300, 35)


## Create Random Forest Model

In [75]:
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=20,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1,
    reg_lambda=1,
    objective='multi:softprob',
    eval_metric='mlogloss',
    random_state=42
)

## Train Model

In [76]:
model.fit(X_train, y_train)

print('Model Training Completed!')

Model Training Completed!


## Make Predictions

In [77]:
train_pred = model.predict(X_train)

test_pred = model.predict(X_test)

## Calculate Accuracy

In [78]:
train_accuracy = accuracy_score(y_train, train_pred)

test_accuracy = accuracy_score(y_test, test_pred)

print('Train Accuracy:', train_accuracy)
print('Test Accuracy:', test_accuracy)

Train Accuracy: 0.9891666666666666
Test Accuracy: 0.9466666666666667


## Classification Report

In [79]:
print(
    classification_report(
        y_test,
        test_pred,
        target_names=target_encoder.classes_
    )
)

                            precision    recall  f1-score   support

         Certified Plumber       1.00      1.00      1.00        33
Customer Support Executive       0.92      0.68      0.78        34
       Data Entry Operator       0.78      0.98      0.87        41
         Junior Accountant       0.97      0.95      0.96        38
       Modern Farm Manager       1.00      1.00      1.00        45
      Retail Store Manager       1.00      1.00      1.00        35
           Sales Associate       0.95      0.95      0.95        41
             Web Developer       1.00      1.00      1.00        33

                  accuracy                           0.95       300
                 macro avg       0.95      0.94      0.95       300
              weighted avg       0.95      0.95      0.95       300



## Save Model Files

In [80]:
dump(model, 'rmf_model.joblib')

dump(education_encoder, 'rmf_education_encoder.joblib')

dump(target_encoder, 'rmf_target_encoder.joblib')

dump(mlb, 'rmf_mlb_encoder.joblib')

dump(training_columns, 'rmf_training_columns.joblib')

print('All files saved successfully!')

All files saved successfully!
